In [1]:
import kagglehub

path = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")

Using Colab cache for faster access to the 'fruits-fresh-and-rotten-for-classification' dataset.


In [2]:
!pip install tensorflow
import tensorflow as tf
import os
train_path = os.path.join(path, "dataset","train")

In [3]:
img_height = 224
img_width = 224
batch_size = 32

In [4]:
import tensorflow as tf

# Load the training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Load the validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 10901 files belonging to 6 classes.
Using 8721 files for training.
Found 10901 files belonging to 6 classes.
Using 2180 files for validation.


In [5]:
# Print class names to verify
original_class_name_for_display = train_ds.class_names
print("Class names:", original_class_name_for_display)

Class names: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


In [6]:
def normalize(image, label):
  image = tf.cast(image/255., tf.float32)
  return image, label

In [7]:
train_ds = train_ds.map(normalize)
val_ds = val_ds.map(normalize)

In [23]:
# Identify rotten fruit class indices

rotten_indices = [
    i for i, name in enumerate(original_class_name_for_display)
    if name.startswith("rotten")
]

print("Rotten class indices:", rotten_indices)


def map_binary(img, label):

    # Convert 6 classes into 2 classes
    binary_label = tf.cast(label >= 3, tf.float32)

    # Add an output dimension
    binary_label = tf.expand_dims(binary_label, axis=-1)

    return img, binary_label

Rotten class indices: [3, 4, 5]


In [24]:
train_ds = train_ds.map(map_binary)
val_ds = val_ds.map(map_binary)

# Data Augmentation For Training **Dataset**

In [25]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2)
])

def augment(img, label):

    image = data_augmentation(img)

    return image, label

train_ds_augmented = train_ds.map(augment)

In [33]:
model = tf.keras.Sequential([

    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, 3)),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [36]:
epochs = 20
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [37]:
history = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[early_stopping]
)

Epoch 1/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 158s 537ms/step - accuracy: 0.8101 - loss: 0.4346 - val_accuracy: 0.9266 - val_loss: 0.2108
Epoch 2/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 126s 459ms/step - accuracy: 0.9139 - loss: 0.2144 - val_accuracy: 0.9454 - val_loss: 0.1404
Epoch 3/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 131s 479ms/step - accuracy: 0.9248 - loss: 0.1899 - val_accuracy: 0.9349 - val_loss: 0.1662
Epoch 4/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 132s 482ms/step - accuracy: 0.9309 - loss: 0.1698 - val_accuracy: 0.9587 - val_loss: 0.1128
Epoch 5/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 137s 464ms/step - accuracy: 0.9400 - loss: 0.1474 - val_accuracy: 0.9569 - val_loss: 0.1162
Epoch 6/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 131s 478ms/step - accuracy: 0.9434 - loss: 0.1303 - val_accuracy: 0.9587 - val_loss: 0.1007
Epoch 7/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 125s 458ms/step - accuracy: 0.9494 - loss: 0.1283 - val_accuracy: 0.9372 - val_loss: 0.1474
Epoch 8/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 125s 457ms/step - accuracy: 0.9546 -

# **Evaluating the Model**

In [38]:
test_path = os.path.join(path, "dataset","test")

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

Found 2698 files belonging to 6 classes.


In [39]:
test_ds = test_ds.map(normalize)
test_ds = test_ds.map(map_binary)

In [40]:
loss, accuracy = model.evaluate(test_ds)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

85/85 ━━━━━━━━━━━━━━━━━━━━ 18s 216ms/step - accuracy: 0.9815 - loss: 0.0490
Test Loss: 0.04900925233960152
Test Accuracy: 0.9814677834510803


In [41]:
model_save_path = 'fruits_classification_model.h5'
model.save(model_save_path)

In [42]:
model_save_path = 'fruits_classification_model.keras'
model.save(model_save_path)